# Your first embedding

## Exercise objectives:

- Run your first RNN for NLP
- Get a first taste of what an embedding is

<hr>

Words are not something you can easily feed to a Neural Network. For this reason, we have to convert them to something more meaningful. 

And this is exactly what _Embeddings_ are for! They map any word onto a vectorial representation (this a fancy way to represent each word with a vector ;) ). For instance, the word `dog` can be represented by the vector $(w_1, w_2, ..., w_n)$ in the embedding space, and we will learn the weights $(w_k)_k$.

So let's just do it.

# The data

❓ **Question** ❓ Let's first load the data. You don't have to understand what is going on in the function, it does not matter here.

⚠️ **Warning** ⚠️ The `load_data` function has a `percentage_of_sentences` argument. Depending on your computer, there are chances that too many sentences will make your compute slow down, or even freeze - your RAM can overflow. For that reason, **you should start with 10% of the sentences** and see if your computer handles it. Otherwise, rerun with a lower number. 

⚠️ **DISCLAIMER** ⚠️ **No need to play _who has the biggest_ (RAM) !** The idea is to get to run your models quickly to prototype. Even in real life, it is recommended that you start with a subset of your data to loop and debug quickly. So increase the number only if you are into getting the best accuracy. 

In [ ]:
###########################################
### Just run this cell to load the data ###
###########################################

import tensorflow_datasets as tfds
from tensorflow.keras.preprocessing.text import text_to_word_sequence

def load_data(percentage_of_sentences=None):
    train_data, test_data = tfds.load(name="imdb_reviews", split=["train", "test"], batch_size=-1, as_supervised=True)

    train_sentences, y_train = tfds.as_numpy(train_data)
    test_sentences, y_test = tfds.as_numpy(test_data)

    # Take only a given percentage of the entire data
    if percentage_of_sentences is not None:
        assert(percentage_of_sentences> 0 and percentage_of_sentences<=100)

        len_train = int(percentage_of_sentences/100*len(train_sentences))
        train_sentences, y_train = train_sentences[:len_train], y_train[:len_train]

        len_test = int(percentage_of_sentences/100*len(test_sentences))
        test_sentences, y_test = test_sentences[:len_test], y_test[:len_test]

    X_train = [text_to_word_sequence(_.decode("utf-8")) for _ in train_sentences]
    X_test = [text_to_word_sequence(_.decode("utf-8")) for _ in test_sentences]

    return X_train, y_train, X_test, y_test

X_train, y_train, X_test, y_test = load_data(percentage_of_sentences=10)

2025-04-10 13:23:23.453280: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-04-10 13:23:23.693022: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-10 13:23:38.849666: W tensorflow/core/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HT

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/25000 [00:00<?, ? examples/s]

...

Shuffling ~/tensorflow_datasets/imdb_reviews/plain_text/1.0.0.incomplete3APRPK/imdb_reviews-test.tfrecord*...:…

Generating unsupervised examples...:   0%|          | 0/50000 [00:00<?, ? examples/s]

Shuffling ~/tensorflow_datasets/imdb_reviews/plain_text/1.0.0.incomplete3APRPK/imdb_reviews-unsupervised.tfrec…

Dataset imdb_reviews downloaded and prepared to ~/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.


2025-04-10 13:26:10.959070: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Now that you have loaded the data, let's check it out!

❓ **Question** ❓ You can play with the data here. In particular, `X_train` and `X_test` are lists of sentences. Let's print some of them, with their respective label stored in `y_train` and `y_test`.

In [3]:
# YOUR CODE HERE
# Print a few examples from the training set
for i in range(5):
    print(f"Sentence {i+1} (Train): {' '.join(X_train[i])}")
    print(f"Label {i+1} (Train): {y_train[i]}")
    print()

# Print a few examples from the test set
for i in range(5):
    print(f"Sentence {i+1} (Test): {' '.join(X_test[i])}")
    print(f"Label {i+1} (Test): {y_test[i]}")
    print()

Sentence 1 (Train): this was an absolutely terrible movie don't be lured in by christopher walken or michael ironside both are great actors but this must simply be their worst role in history even their great acting could not redeem this movie's ridiculous storyline this movie is an early nineties us propaganda piece the most pathetic scenes were those when the columbian rebels were making their cases for revolutions maria conchita alonso appeared phony and her pseudo love affair with walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning i am disappointed that there are movies like this ruining actor's like christopher walken's good name i could barely sit through it
Label 1 (Train): 0

Sentence 2 (Train): i have been known to fall asleep during films but this is usually due to a combination of things including really tired being warm and comfortable on the sette and having just eaten a lot however on this occasion i fell asleep because the fil

**LABELS**: the task is a binary classification problem:
- label 0️⃣ corresponds to a <font color=red>negative</font> movie review
- label 1️⃣ corresponds to a <font color=green>positive</font> movie review

**INPUTS**: 
- 🧹 The data has been partially cleaned! So you don't have to worry about it in this exercise. 
- ❗️ But don't forget this step in real-life challenges. 

Remember that words are not computer-compatible materials? You have to tokenize them!

❓ **Question** ❓ Run the following cell to tokenize your sentences

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# This initializes a Keras utilities that does all the tokenization for you
tokenizer = Tokenizer()

# The tokenization learns a dictionary that maps a token (integer) to each word
# It can be done only on the train set - we are not supposed to know the test set!
# This tokenization also lowercases your words, apply some filters, and so on - you can check the doc if you want
tokenizer.fit_on_texts(X_train)

# We apply the tokenization to the train and test set
X_train_token = tokenizer.texts_to_sequences(X_train)
X_test_token = tokenizer.texts_to_sequences(X_test)

❓ **Question** ❓ Print some of the tokenized sentences to be sure you got what you expected

In [7]:
# YOUR CODE HERE
# Print a few tokenized examples from the test set
for i in range(5):
    print(f"Tokenized Sentence {i+1} (Test): {X_test_token[i]}")
    print()

Tokenized Sentence 1 (Test): [48, 24, 100, 12, 95, 3403, 15, 842, 4138, 10, 13, 276, 4, 1, 761, 351, 15, 3789, 1307, 17749, 15, 562, 11419, 6248, 683, 5, 12, 1183, 422, 588, 17461, 2, 324, 3653, 19, 251, 3, 14, 9247, 3, 14, 2105, 14, 99, 4, 1, 2823, 97, 9, 728, 1228, 11, 241, 230, 9, 210, 1, 385, 6182, 3, 58, 87, 9, 91, 101, 9, 1228, 174, 11, 241, 34, 5, 1032, 698, 6, 4228, 17461, 2, 324, 6, 34, 8215, 385, 4, 1813, 12, 26, 60, 25, 5, 753, 178, 16, 2, 960, 4, 11, 227, 3, 74, 2, 350, 31, 350, 5948, 4, 10, 5, 1275, 1458, 1, 28332, 3, 27148, 4, 10, 159, 345, 6, 1440, 4274, 2, 871, 1419, 4, 2, 244, 2388, 160, 3, 1, 370, 29, 185, 24, 1294, 224, 666, 4, 1, 117, 1, 338, 1157, 11610, 26, 14635, 782, 37, 2, 19, 35, 11, 17461, 2, 324, 6, 2, 19, 594, 57, 45, 222]

Tokenized Sentence 2 (Test): [2, 655, 725, 4, 2, 178, 2762, 9510, 1, 14408, 12, 8826, 18783, 13, 501, 5, 3802, 8, 115, 501, 5, 395, 2, 1874, 16549, 16, 2, 5483, 4, 5675, 14, 33, 7223, 37, 23, 4634, 1363, 4, 19, 251, 10, 13, 33, 12447, 69

The dictionary that maps each word to a token can be accessed with `tokenizer.word_index`
    
❓ **Question** ❓ Add a `vocab_size` variable that stores the number of different words (=tokens) in the train set. This is called the _size of the vocabulary_.

In [9]:
# YOUR CODE HERE
vocab_size = len(tokenizer.word_index)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 30419


Your `X_train_token` and `X_test_token` contain sequences of different lengths.

<img src="padding.png" alt='Word2Vec' width="700px" />

However, a neural network has to have a tensor as input. For this reason, you have to pad your data.

❓ **Question** ❓  Pad your data with the `pad_sequences` function (documentation [here](https://www.tensorflow.org/api_docs/python/tf/keras/preprocessing/sequence/pad_sequences)). Do not forget about the `dtype` and `padding` keywords (but do not use `maxlen` here).

In [11]:
# YOUR CODE HERE
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Pad the sequences
X_train_padded = pad_sequences(X_train_token, dtype='int32', padding='post')
X_test_padded = pad_sequences(X_test_token, dtype='int32', padding='post')

print(f"Shape of X_train_padded: {X_train_padded.shape}")
print(f"Shape of X_test_padded: {X_test_padded.shape}")

Shape of X_train_padded: (2500, 1164)
Shape of X_test_padded: (2500, 988)


# The RNN

Let's now feed this data to a Recurrent Neural Network.

❓ **Question** ❓ Write a model that has:
- an embedding layer whose `input_dim` is the size of your vocabulary (= your `vocab_size`), and whose `output_dim` is the size of the embedding space you want to have
- a RNN (SimpleRNN, LSTM, GRU) layer
- a Dense layer
- an output layer

⚠️ **Warning** ⚠️ Here, you don't need a masking layer. Why? Because `layers.Embedding` has a argument to do that directly, which you have to set with `mask_zero=True`. That also means that your data **HAS TO** be padded with **0** (which is the default behavior). See the [documentation](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding#example_2) to understand how it **impacts** the `input_dim`.

<details>
    <summary>💡 Hint</summary>

`input_dim` should equal size of vocabulary + 1

</details>

Compile it with the appropriate arguments

In [13]:
# YOUR CODE HERE
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Define the embedding dimension
embedding_dim = 128

# Build the model
model = Sequential([
    Embedding(input_dim=vocab_size + 1, output_dim=embedding_dim, mask_zero=True),
    LSTM(64, return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary classification
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, None, 128)         3893760   
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3,945,281
Trainable params: 3,945,281
Non-trainable params: 0
_________________________________________________________________


❓ **Question** ❓ Look at the number of parameters in your RNN. 

In [17]:
# YOUR CODE HERE
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, None, 128)         3893760   
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 1)                 33        
                                                                 
Total params: 3,945,281
Trainable params: 3,945,281
Non-trainable params: 0
_________________________________________________________________


❓ Double-check that the number of parameters in your embedding layer is equal to the (number of words in your vocabulary + 1 for the masking value) $\times$  the dimension of your embedding.

In [19]:
# YOUR CODE HERE
# Calculate the expected number of parameters in the embedding layer
expected_params = (vocab_size + 1) * embedding_dim
print(f"Expected number of parameters in the embedding layer: {expected_params}")

# Retrieve the actual number of parameters from the model summary
actual_params = model.layers[0].count_params()
print(f"Actual number of parameters in the embedding layer: {actual_params}")

# Verify if they match
assert expected_params == actual_params, "The number of parameters does not match!"
print("The number of parameters matches!")

Expected number of parameters in the embedding layer: 3893760
Actual number of parameters in the embedding layer: 3893760
The number of parameters matches!


❓ **Question** ❓ Start fitting your model with 20 epochs, with an early stopping criterion whose patience is equal to 4.

⚠️ **Warning** ⚠️ You might see that it takes a lot of time! 

**So stop it after a couple of iterations!**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

# Fit the model
history = model.fit(
    X_train_padded,
    y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=[early_stopping]
)

Epoch 1/20
63/63 [==============================] - 67s 1s/step - loss: 0.6587 - accuracy: 0.6090 - val_loss: 0.5756 - val_accuracy: 0.7240
Epoch 2/20
63/63 [==============================] - 70s 1s/step - loss: 0.2932 - accuracy: 0.8855 - val_loss: 0.4023 - val_accuracy: 0.8100
Epoch 3/20
63/63 [==============================] - 74s 1s/step - loss: 0.0609 - accuracy: 0.9845 - val_loss: 0.8009 - val_accuracy: 0.8040
Epoch 4/20
63/63 [==============================] - 70s 1s/step - loss: 0.0195 - accuracy: 0.9955 - val_loss: 0.5266 - val_accuracy: 0.8040
Epoch 5/20
63/63 [==============================] - 70s 1s/step - loss: 0.0167 - accuracy: 0.9960 - val_loss: 0.7432 - val_accuracy: 0.7960
Epoch 6/20
63/63 [==============================] - 73s 1s/step - loss: 0.0066 - accuracy: 0.9985 - val_loss: 0.8358 - val_accuracy: 0.7960


In [21]:
# YOUR CODE HERE


Let's not waste too much time just staring at our screen or having coffees. It is too early to start having breaks ;)

❓ **Question** ❓ We will reduce the computational time. To start, let's first look at how many words there are in the different sentences of your train set (Just run the following cell).

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

def plot_hist(X):
    len_ = [len(_) for _ in X]
    plt.hist(len_)
    plt.title('Histogram of the number of sentences that have a given number of words')
    plt.show()

plot_hist(X_train)


You will probably see that 90 to 95% of your sentences have less than 300 words. And very few have more than 1000.

However, as you didn't use `maxlen` in your padding above, your input tensor has a dimension equal to the length of the sentence that has the maximum number of words.

Now, let's look at how this affects the padding: 


<img src="tensor_size.png" alt='Word2Vec' width="700px" />

Because of a few of very long sentences, one dimension of your tensor is equal to around 1000. However, most of the sentences with ~200 words have just padded values that are useless.

So your tensor is mostly useless information, which still adds time to the training process.

But what if you pad the data to a maximum length (`maxlen`) of say 200 (words)?
- First, that would increase the convergence and you would not need to stare at your screen while waiting for the algorithm to converge
- But in essence, do you really lose that much information? Do you think that you often need more than 200 words (up to 1000) to tell whether or not a sentence is positive of negative?

❓ **Question** ❓ For all these reasons, re-do your padding using the `maxlen` keyword and retrain the model!  See how much faster it is now - without hurting the performance ;)

In [ ]:
# YOUR CODE HERE

🏁 Nice, you are now able to use `Tokenizer` and `pad_sequences`

💾 Don't forget to git add/commit/push your notebook...

🚀 ... and move on to the next challenge!